## Agent 的高级用法
### 4 ：流式输出及模式

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式

)



In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
      """
      查询客户基本信息

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户基本信息的字典，如姓名、等级、加入日期等
      """
      # 模拟数据库查询
      return {"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
      """
      查询客户订单历史

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户订单历史的字典，如总订单数、总花费等
      """
      return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
      """
      获取当前可用促销活动

      Returns:
      包含当前可用促销活动的字典，如活动名称、有效日期等
      """
      return {
            "promotions": [" 老用户优惠 ", " 会员专属折扣 "],
            "valid_until": "2027-01-31",
      }

      # 创建客户服务 Agent


customer_service_agent = create_agent(
      model=model,
      tools=[query_customer_data, check_order_history, get_current_promotions],
)

for chunk in customer_service_agent.stream(
        {
              "messages": [
                    {
                          "role": "user",
                          "content": " 查询客户 ID 为 CUST123456 的个人信息、历史订单和可用优惠 ",
                    }
              ]
        },
        stream_mode="values",
):
      rprint(chunk)
      print("-" * 50)

### 2.2 updates 输出模式
这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控
Agent 执行进度，例如观察 Agent 决定调用工具、工具执行结果等步骤。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
      """
      查询客户基本信息

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户基本信息的字典，如姓名、等级、加入日期等
      """
      # 模拟数据库查询
      return {"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
      """
      查询客户订单历史

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户订单历史的字典，如总订单数、总花费等
      """
      return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
      """
      获取当前可用促销活动

      Returns:
      包含当前可用促销活动的字典，如活动名称、有效日期等
      """
      return {
            "promotions": [" 老用户优惠 ", " 会员专属折扣 "],
            "valid_until": "2027-01-31",
      }

      # 创建客户服务 Agent


customer_service_agent = create_agent(
      model=model,
      tools=[query_customer_data, check_order_history, get_current_promotions],
)

for chunk in customer_service_agent.stream(
        {
              "messages": [
                    {
                          "role": "user",
                          "content": " 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ",
                    }
              ]
        },
        stream_mode="updates",
):
      rprint(chunk)
      print("-" * 50)

### messages 输出模式

In [15]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
      """
      查询客户基本信息

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户基本信息的字典，如姓名、等级、加入日期等
      """
      # 模拟数据库查询
      return {"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
      """
      查询客户订单历史

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户订单历史的字典，如总订单数、总花费等
      """
      return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
      """
      获取当前可用促销活动

      Returns:
      包含当前可用促销活动的字典，如活动名称、有效日期等
      """
      return {
            "promotions": [" 老用户优惠 ", " 会员专属折扣 "],
            "valid_until": "2027-01-31",
      }

      # 创建客户服务 Agent


customer_service_agent = create_agent(
      model=model,
      tools=[query_customer_data, check_order_history, get_current_promotions],
)

# 其他工具代码同上，保持不变
# ... ...
for chunk in customer_service_agent.stream(
        {
              "messages": [
                    {
                          "role": "user",
                          "content": " 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ",
                    }
              ]
        },
        stream_mode="messages",
):
      print(chunk)
      print("-" * 50)

      # print(chunk[0].content,end="",flush=True)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--01a1daa1-a739-7192-8b96-5f90e29d0ba1', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:eb398508-2311-cf5d-2634-91dc76d3d001', 'checkpoint_ns': 'model:eb398508-2311-cf5d-2634-91dc76d3d001', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None, 'ls_integration': 'langchain_chat_model', 'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.2.12', 'langchain-openai': '1.7.4', 'langchain-deepseek': '1.1.1'}})
--------------------------------------------------
(AIMessageChunk(content='我来', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--01a1daa1-a739-7192-8b96-5f90e29d0ba1', tool_calls=[], invalid_too

###  tasks 输出模式
该模式会输出当前 task 任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生
命周期。

In [16]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
      """
      查询客户基本信息

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户基本信息的字典，如姓名、等级、加入日期等
      """
      # 模拟数据库查询
      return {"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
      """
      查询客户订单历史

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户订单历史的字典，如总订单数、总花费等
      """
      return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
      """
      获取当前可用促销活动

      Returns:
      包含当前可用促销活动的字典，如活动名称、有效日期等
      """
      return {
            "promotions": [" 老用户优惠 ", " 会员专属折扣 "],
            "valid_until": "2027-01-31",
      }

      # 创建客户服务 Agent


customer_service_agent = create_agent(
      model=model,
      tools=[query_customer_data, check_order_history, get_current_promotions],
)

# 其他工具代码同上，保持不变
# ... ...
for chunk in customer_service_agent.stream(
        {
              "messages": [
                    {
                          "role": "user",
                          "content": " 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ",
                    }
              ]
        },
        stream_mode="tasks",
):
      print(chunk)
      print("-" * 50)

      # print(chunk[0].content,end="",flush=True)

{'id': '773ad07b-84ac-24e9-0fae-2a8c5178027a', 'name': 'model', 'input': {'messages': [HumanMessage(content=' 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ', additional_kwargs={}, response_metadata={}, id='0218294d-6393-4250-9f76-98da293c43d0')]}, 'triggers': ('branch:to:model',)}
--------------------------------------------------
{'id': '773ad07b-84ac-24e9-0fae-2a8c5178027a', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content="I'll query the customer information and current promotions for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 453, 'total_tokens': 555, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 197}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe01647990

###  debug 输出模式
该模式与 tasks 模式类似，比 task 模式多输出任务步骤、时间戳、 task 类型（ task/task_result ），该模式
用于调试、监控 task 任务的生命周期。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
      """
      查询客户基本信息

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户基本信息的字典，如姓名、等级、加入日期等
      """
      # 模拟数据库查询
      return {"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
      """
      查询客户订单历史

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户订单历史的字典，如总订单数、总花费等
      """
      return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
      """
      获取当前可用促销活动

      Returns:
      包含当前可用促销活动的字典，如活动名称、有效日期等
      """
      return {
            "promotions": [" 老用户优惠 ", " 会员专属折扣 "],
            "valid_until": "2027-01-31",
      }

      # 创建客户服务 Agent


customer_service_agent = create_agent(
      model=model,
      tools=[query_customer_data, check_order_history, get_current_promotions],
)

# 其他工具代码同上，保持不变
# ... ...
for chunk in customer_service_agent.stream(
        {
              "messages": [
                    {
                          "role": "user",
                          "content": " 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ",
                    }
              ]
        },
        stream_mode="debug",
):
      print(chunk)
      print("-" * 50)

      # print(chunk[0].content,end="",flush=True)

### checkpoints 输出模式
该模式中，每当检查点（ checkpoint ）被创建时会触发输出，输出包含检查点中的状态，用于需要状态
持久化、工作流恢复或分布式执行跟踪的高级场景。

In [22]:
from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
      """
      查询客户基本信息

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户基本信息的字典，如姓名、等级、加入日期等
      """
      # 模拟数据库查询
      return {"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
      """
      查询客户订单历史

      Args:
      customer_id: 客户 ID ，用于唯一标识客户

      Returns:
      包含客户订单历史的字典，如总订单数、总花费等
      """
      return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
      """
      获取当前可用促销活动

      Returns:
      包含当前可用促销活动的字典，如活动名称、有效日期等
      """
      return {
            "promotions": [" 老用户优惠 ", " 会员专属折扣 "],
            "valid_until": "2027-01-31",
      }


# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建 Agent
customer_service_agent = create_agent(
      model=model,
      tools=[query_customer_data, check_order_history, get_current_promotions],
      checkpointer=checkpointer,  # 启用检查点
)

# 3. 创建唯一的会话 ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用 Agent
checkpoint_count = 0

# 使用 checkpoints 模式进行流式监控
for chunk in customer_service_agent.stream(
        {
              "messages": [
                    {
                          "role": "user",
                          "content": " 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ",
                    }
              ]
        },
        config=config,
        stream_mode="checkpoints",
):
      checkpoint_count += 1
      print(f" 检查点 #{checkpoint_count}")
rprint(chunk)
print("-" * 50)

 检查点 #1
 检查点 #2
 检查点 #3
 检查点 #4
 检查点 #5


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e03f9-8584-6c21-8003-58dff1720fc2'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e03f9-74cd-6455-8002-40608ef5c05b'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content=' 查询客户 ID 为 CUST123456 的完整信息和可用优惠 ',
                additional_kwargs={},
                response_metadata={},
                id='14162ee4-eeb8-4f5f-a513-f34f4b79343b'
            ),
            AIMessage(
                content="I'll query the customer information, order history, and current promotions for 
CUST123456.",
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 109,
                        'prompt_tokens': 453,
                        'total_tokens': 562,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 256,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 256,
                        'prompt_cache_miss_tokens': 197
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '816a5b08-6d5e-49a7-906b-b50b1dad3bf5',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1daa9-a714-719f-999e-f5cbb84d3e37-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_00_bXF2su7tHYDOcNngXFqQ2421',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_01_wjjODfFI3MlrBC00igUG3983',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_ya9xWgbJBZZDDLJhFD471955',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 453,
                    'output_tokens': 109,
                    'total_tokens': 562,
                    'input_token_details': {'cache_read': 256},
                    'output_token_details': {}
                }
            ),
            ToolMessage(
                content='{"name": " 张三 ", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='eb57c514-efa7-4bc2-9f1c-158ef35ceb4f',
                tool_call_id='call_00_bXF2su7tHYDOcNngXFqQ2421'
            ),
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='e04bd373-9250-4985-8efe-37f15eebde9e',
                tool_call_id='call_01_wjjODfFI3MlrBC00igUG3983'
            ),
            ToolMessage(
                content='{"promotions": [" 老用户优惠 ", " 会员专属折扣 "], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='89013182-cf44-407f-9c

--------------------------------------------------


###  custom 输出模式
开发者通过 get_stream_writer 在工具或节点内部 自定义发送的数据 ，用于 输出 业务逻辑相关的进
度信息（如 “ 已处理 10/100 条记录 ” ）、自定义日志或指标。

In [18]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time


# **举例：** 生成销售报告和库存报告 Agent


@tool
def generate_sales_report() -> str:
      """生成销售报告"""
      writer = get_stream_writer()

      writer({"type": " 生成销售报告 ", "message": " 开始生成销售报告 "})

      # 模拟数据处理
      for i in range(1, 4):
            time.sleep(0.5)
            writer(
                  {"type": " 生成销售报告 ", "message": f" 生成销售报告进度百分比： {i * 25}%"}
            )

      writer({"type": " 生成销售报告 ", "message": " 报告生成完成 "})

      return f" 销售报告：总收入 150 万元，同比增长 12%"


@tool
def generate_inventory_report() -> str:
      """生成库存报告"""
      writer = get_stream_writer()
      writer(" 开始库存分析 ...")
      time.sleep(0.5)
      writer(" 检查当前库存量 ...")
      time.sleep(0.5)
      writer(" 生成库存报告 ...")

      return " 当前库存量为 10000 件，库存充足，无异常 "


# 创建报告生成 agent
reporting_agent = create_agent(
      model=model, tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
        {"messages": [{"role": "user", "content": " 生成销售报告和库存报告 "}]},
        stream_mode="custom",
):
      print(chunk)
      print("-" * 50)

 开始库存分析 ...
--------------------------------------------------
{'type': ' 生成销售报告 ', 'message': ' 开始生成销售报告 '}
--------------------------------------------------
 检查当前库存量 ...
--------------------------------------------------
{'type': ' 生成销售报告 ', 'message': ' 生成销售报告进度百分比： 25%'}
--------------------------------------------------
 生成库存报告 ...
--------------------------------------------------
{'type': ' 生成销售报告 ', 'message': ' 生成销售报告进度百分比： 50%'}
--------------------------------------------------
{'type': ' 生成销售报告 ', 'message': ' 生成销售报告进度百分比： 75%'}
--------------------------------------------------
{'type': ' 生成销售报告 ', 'message': ' 报告生成完成 '}
--------------------------------------------------
